<a href="https://colab.research.google.com/github/mgracemborgonia/Linear-Regression-Health-Costs-Calculator-FCC/blob/main/Copy_of_fcc_predict_health_costs_with_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Import libraries. You may or may not use all of these.
!pip install -q git+https://github.com/tensorflow/docs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
  # %tensorflow_version only exists in Colab.
  %tensorflow_version 2.x
except Exception:
  pass
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers

import tensorflow_docs as tfdocs
import tensorflow_docs.plots
import tensorflow_docs.modeling

In [ ]:
# Import data
!wget https://cdn.freecodecamp.org/project-data/health-costs/insurance.csv
dataset = pd.read_csv('insurance.csv')
dataset.tail()

In [ ]:
len(dataset)

In [ ]:
df = pd.get_dummies(dataset, columns=["sex", "smoker", "region"], prefix=None, prefix_sep="_", drop_first=True, dtype=int)
df.tail()

In [ ]:
train_dataset = df.sample(frac=0.8, random_state=42)
train_labels = train_dataset.pop("expenses")
print(train_dataset.shape)
print(train_labels.shape)
print(len(train_dataset))
train_labels.tail()

In [ ]:
test_dataset = df.sample(frac=0.2, random_state=42)
test_labels = test_dataset.pop("expenses")
print(test_dataset.shape)
print(test_labels.shape)
print(len(test_dataset))
test_labels.tail()

In [ ]:
normalizer = layers.Normalization(axis=-1)
adapt_data= np.array(train_dataset)
normalizer.adapt(adapt_data)
model = keras.Sequential()
model.add(normalizer)
model.add(layers.Dense(16, activation='relu'))
model.add(layers.Dropout(0.2))
model.add(layers.Dense(8, activation='relu'))
model.add(layers.Dense(1))

In [ ]:
model.compile(
    optimizer=tf.optimizers.Adam(learning_rate=0.1),
    loss=tf.losses.MeanAbsoluteError(name='mean_absolute_error'),
    metrics=['mae', 'mse']
)
model.summary()

In [ ]:
model.fit(
    train_dataset,
    train_labels,
    epochs=100,
    validation_split=0.5,
    verbose=0
)

In [ ]:
# RUN THIS CELL TO TEST YOUR MODEL. DO NOT MODIFY CONTENTS.
# Test model by checking how well the model generalizes using the test set.
loss, mae, mse = model.evaluate(test_dataset, test_labels, verbose=2)

print("Testing set Mean Abs Error: {:5.2f} expenses".format(mae))

if mae < 3500:
  print("You passed the challenge. Great job!")
else:
  print("The Mean Abs Error must be less than 3500. Keep trying.")

# Plot predictions.
test_predictions = model.predict(test_dataset).flatten()

a = plt.axes(aspect='equal')
plt.scatter(test_labels, test_predictions)
plt.xlabel('True values (expenses)')
plt.ylabel('Predictions (expenses)')
lims = [0, 50000]
plt.xlim(lims)
plt.ylim(lims)
_ = plt.plot(lims,lims)
